In [1]:
import json
import os
import httpx2
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key and openrouter_api_key.startswith("sk-or-v1-"):
    print(f"OpenRouter API key exists and begins {openrouter_api_key[:9]}")
else:
    print("No OpenRouter API key found. Please check your .env file (Setup part 4).")

OpenRouter API key exists and begins sk-or-v1-


In [4]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = "openrouter/free"

# Retry up to 5 times when a free model is busy
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

In [5]:
messages = [{"role": "user", "content": "Hi there!"}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Answered by dots-studio/dots-3-note-preview:free




Hi there! How can I help you today?

In [6]:
MAZE_URL = "https://maze.edwarddonner.com"

def read_web_page(url):
    """Read a web page in the maze and return the text on the page, without the HTML"""
    print(f"Reading web page: {url}")
    if not url.startswith(MAZE_URL):
        return f"Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with {MAZE_URL}"
    response = httpx2.get(url, timeout=20)
    page = BeautifulSoup(response.text, "html.parser")
    return page.body.get_text(separator="\n", strip=True)

In [7]:
print(read_web_page("https://maze.edwarddonner.com"))

Reading web page: https://maze.edwarddonner.com
The Maze
This page is for an AI that can read web pages. Your goal is to find the way out of a small maze and bring back the message that is waiting there. Every location in the maze is a web page that describes the room and lists its exits.
Step 1 is the entrance:
Location 1
https://maze.edwarddonner.com/1
Start with the maze now. Open the entrance, then follow the exits one page at a time. Some passages are dead ends, so go back and try another way when you reach one.
Keep the person you are working with updated as you go. After each location, say where you are and what you found there, and show a mini-map of the maze as you have seen it so far, drawn in text with the rooms you have visited and the exits you know about. When you reach the way out, give the message and a final map of your whole journey.
If you are a person reading this, give this address to ChatGPT or Claude and ask it to do what the page says.


In [8]:
print(read_web_page("https://edwarddonner.com"))

Reading web page: https://edwarddonner.com
Sorry, this tool can only read pages in the maze. Please ask for a URL that starts with https://maze.edwarddonner.com


In [9]:
read_web_page_json = {
    "type": "function",
    "function": {
        "name": "read_web_page",
        "description": "Read a web page and return the text on the page",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {"type": "string", "description": "The full URL of the web page to request"},
            },
            "required": ["url"],
        },
    },
}

tools = [read_web_page_json]

In [10]:
def handle_tool_call(tool_call):
    """Run the tool that the model asked for and package the result as a message"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        result = read_web_page(**arguments)
    except Exception as error:
        result = f"The tool call failed: {error}"
    return {"role": "tool", "tool_call_id": tool_call.id, "content": result}

In [11]:
messages = [{"role": "user", "content": "Please read https://maze.edwarddonner.com and tell me what it says. Just describe the page for now, without going into the maze."}]

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
message = response.choices[0].message

print(f"Answered by {response.model}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(message.tool_calls)

Answered by minimax/minimax-m3.1-flash:free
Finish reason: tool_calls
[ChatCompletionMessageFunctionToolCall(id='call_function_ikkq9xplilv9_1', function=Function(arguments='{"url": "https://maze.edwarddonner.com"}', name='read_web_page'), type='function', index=0)]


In [12]:
messages.append(message)
for tool_call in message.tool_calls:
    messages.append(handle_tool_call(tool_call))

response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)

print(f"Answered by {response.model}")
display(Markdown(response.choices[0].message.content))

Reading web page: https://maze.edwarddonner.com
Answered by poolside/laguna-s-2.1:free


<IPython.core.display.Markdown object>

Part 2 - Harness Loops  

In [13]:
SYSTEM_PROMPT = """
You are an agent that completes tasks by using your tools, one step at a time.
Your replies are displayed as Markdown, so format them in Markdown with headings, bold text and lists.
Put any map or diagram in a code block so that it lines up. Do not use emojis.
"""

In [14]:
def run_agent(task, max_steps=25):
    """Call the model in a loop, running the tool whenever it asks, until it gives a final answer"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": task},
    ]
    for step in range(1, max_steps + 1):
        response = openrouter.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        message = response.choices[0].message
        print(f"Step {step} answered by {response.model}")

        text = (message.content or "").strip()
        if text:
            display(Markdown(text))

        if message.tool_calls:
            messages.append(message)
            for tool_call in message.tool_calls:
                messages.append(handle_tool_call(tool_call))
        elif text:
            return
        else:
            print("The reply was empty, so we ask again")

    print("The agent ran out of steps")

In [15]:
run_agent("Have a look at https://maze.edwarddonner.com and do what it says.")

Step 1 answered by recursive/flint-v1.0:free
Reading web page: https://maze.edwarddonner.com
Step 2 answered by dots-studio/dots-3-note-preview:free
Reading web page: https://maze.edwarddonner.com/1
Step 3 answered by nvidia/nemotron-3-ultra-550b-a55b:free


**Location 1 (Entrance)**
- Exits: North → 14, East → 27

**Map so far:**
```
    [14]?
     |
[1]--+
     |
    [27]?
```

Let me explore North first to location 14.

Reading web page: https://maze.edwarddonner.com/14
Step 4 answered by minimax/minimax-m3.1-flash:free


**Location 14** — damp corridor with tally marks scratched in the stone.
- Exits: North → 62, South → 1 (back to entrance)

**Map so far:**
```
        [62]?
         |
        [14]
         |
[27]? -- [1]
```

Reading web page: https://maze.edwarddonner.com/62
Step 5 answered by cohere/north-mini-code:free
Reading web page: https://maze.edwarddonner.com/39
Step 6 answered by poolside/laguna-s-2.1:free


...

Back at **39** — both exits accounted for (71 = dead end, South = back). Heading back to **62** to try the East exit to **48**.

Reading web page: https://maze.edwarddonner.com/48
Step 9 answered by poolside/laguna-s-2.1:free
Reading web page: https://maze.edwarddonner.com/83
Step 10 answered by cohere/north-mini-code:free
Reading web page: https://maze.edwarddonner.com/25
Step 11 answered by recursive/flint-v1.0:free


**Location 83** — round chamber, three ways out. Cool air moving through.
- Exits: North → 25, East → 56, West → 48

**Location 25** — stone staircase going up, daylight at top.
- Exits: East → 35, South → 83

**Map so far:**
```
            [39]
             |
[48]? -- [62] -- [1]
             |
             [83] -- [25] -- [35]?
                  |
                  [39]...
```
Let me try East from 25 to 35.

Reading web page: https://maze.edwarddonner.com/35
Step 12 answered by nvidia/nemotron-3.5-lightning:free


**Message**

```
Welcome to the start of the course! Prepare for 3 intensive weeks ahead.
```

**Final map of the maze journey**

```
[1]  Entrance
   | \
   |  [27] unexplored exit
   | \
   v  v
[14] Damp corridor (south from 1)
   |
   |
[62] Junction, low ceiling, faint hum
 / | \
/  |  \
[39] [48] [14] back to entrance
| \   |
|  \  |
|   [71] Dead end (broom, dead end)
|     |
|     └─► back to 39
|          (explored and ruled out)
v
[48] Narrow bridge over dry channel (hum louder)
|
v
[83] Round chamber, cool air moving through
 / | \
/ |  \
[25] [56] unexplored east from 83
| \
/  [35] Exit – daylight at top
 \
  └─► South back to 83

**Route taken (the path that led out)**
1 → 14 → 62 → 39 → 71 (dead end) → back to 39 → back to 62 → 48 → 83 → 25 → 35 (exit)

**Summary**
- Visited rooms: 1, 14, 62, 39, 71 (dead end), 48, 83, 25, 35 (exit).  
- Dead end encountered: 71 (broom‑filled cupboard).  
- The final exit (location 35) opened to daylight with the welcoming message above.